## Задание 1
Используя Apache Spark (MapReduce), реализовать программу для подсчета количества слов в тексте, которые представляет собой некоторое количество строк, единственный разделитель между словами – пробел.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("WordCount") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
sample_text = '''
    This is a sample text for word counting in PySpark
    PySpark is a powerful tool for big data processing
    This sample demonstrates how to count words in text
'''

text_lines = sample_text.strip().split('\n')
text = sc.parallelize(text_lines)

words = text.flatMap(lambda line: line.strip().split())

# Map each word to a key-value pair (word, 1) and reduce by key
word_counts = words.map(lambda word: (word.lower(), 1)) \
                   .reduceByKey(lambda a, b: a + b)


In [4]:
counts = word_counts.collect()

In [5]:
for word, count in counts:
    print(f"{word}: {count}")

this: 2
sample: 2
text: 2
for: 2
word: 1
counting: 1
powerful: 1
tool: 1
big: 1
how: 1
to: 1
words: 1
is: 2
a: 2
in: 2
pyspark: 2
data: 1
processing: 1
demonstrates: 1
count: 1


## Задание 2
Используя Apache Spark (DataFrame), реализовать программу для подсчета количества слов в тексте, которые представляет собой некоторое количество строк, единственный разделитель между словами – пробел.



In [6]:
# For DataFrame operations
from pyspark.sql.functions import explode, split, lower, col, count

In [7]:
# Use the same sample text as Task 1
sample_text = '''
    This is a sample text for word counting in PySpark
    PySpark is a powerful tool for big data processing
    This sample demonstrates how to count words in text
'''

In [8]:
# Create a DataFrame from the sample text
lines_df = spark.createDataFrame(
    [(line,) for line in sample_text.strip().split('\n')],
    ["line"]
)

In [9]:
# Split each line into words and explode to create a row for each word
words_df = lines_df.select(
    explode(split(col("line"), " ")).alias("word")
)

In [10]:
# Filter out empty words (if any) and convert to lowercase
words_df = words_df.filter(col("word") != "").select(lower(col("word")).alias("word"))

In [11]:
# Count the occurrences of each word
word_counts_df = words_df.groupBy("word").count().orderBy("count", ascending=False)

In [12]:
# Show the results
word_counts_df.show()

+------------+-----+
|        word|count|
+------------+-----+
|         for|    2|
|          in|    2|
|          is|    2|
|     pyspark|    2|
|      sample|    2|
|        text|    2|
|           a|    2|
|        this|    2|
|    counting|    1|
|        word|    1|
|         how|    1|
|    powerful|    1|
|       words|    1|
|       count|    1|
|        data|    1|
|        tool|    1|
|demonstrates|    1|
|  processing|    1|
|          to|    1|
|         big|    1|
+------------+-----+



## Задание 3
Доработайте программу из задания 2: учитывайте, что одно слово может быть записано в разном регистре (Word, word, WORD – одно слово), в тексте могут присутствовать знаки пунктуации, а также то, что количество пробелов между словами может быть больше одного.

На основе полученного набора данных, найдите дополнительно:

топ-10 самый часто встречаемых слов;
слово, которое встречается чаще всего;
слово, которое встречается реже всего;
среднюю встречаемость слов в тексте;
общее число уникальных слов;
общее число слов в тексте.


In [13]:
from pyspark.sql.functions import explode, split, lower, col, count, regexp_replace, avg, sum as spark_sum

In [14]:
# Пример текста для подсчета слов
sample_text = '''
    This is a sample text, for word counting in PySpark!
    PySpark is a powerful   tool for big data processing.
    This sample demonstrates how to count words in text...
    Some words like Word, WORD, and word should be counted as the same word.
'''

In [15]:
# Создание DataFrame из текста
lines_df = spark.createDataFrame(
    [(line,) for line in sample_text.strip().split('\n')],
    ["line"]
)

In [16]:
# Очистка текста: приведение к нижнему регистру и удаление знаков пунктуации
cleaned_df = lines_df.withColumn(
    "cleaned_line",
    regexp_replace(lower(col("line")), "[^a-z0-9\\s]", "")
)

In [17]:
# Разделение каждой строки на слова и создание строки для каждого слова
# Использование split с шаблоном регулярного выражения для обработки нескольких пробелов
words_df = cleaned_df.select(
    explode(split(col("cleaned_line"), "\\s+")).alias("word")
)

In [18]:
# Фильтрация пустых слов (если таковые есть)
words_df = words_df.filter(col("word") != "")

# Подсчет количества вхождений каждого слова
word_counts_df = words_df.groupBy("word").count().orderBy("count", ascending=False)

# Отображение результатов
print("Word counts:")
word_counts_df.show()

Word counts:
+----------+-----+
|      word|count|
+----------+-----+
|      word|    5|
|       for|    2|
|        in|    2|
|        is|    2|
|   pyspark|    2|
|    sample|    2|
|      text|    2|
|         a|    2|
|      this|    2|
|     words|    2|
|  counting|    1|
|  powerful|    1|
|      data|    1|
|      tool|    1|
|processing|    1|
|       big|    1|
|      some|    1|
|       how|    1|
|     count|    1|
|        be|    1|
+----------+-----+
only showing top 20 rows



In [19]:
# Найти топ-10 самых часто встречаемых слов
print("\nТоп-10 самых часто встречаемых слов:")
word_counts_df.limit(10).show()

# Найти самое часто встречаемое слово
most_frequent = word_counts_df.limit(1)
print("\nСамое часто встречаемое слово:")
most_frequent.show()


Топ-10 самых часто встречаемых слов:
+-------+-----+
|   word|count|
+-------+-----+
|   word|    5|
|    for|    2|
|     in|    2|
|     is|    2|
|pyspark|    2|
| sample|    2|
|   text|    2|
|      a|    2|
|   this|    2|
|  words|    2|
+-------+-----+


Самое часто встречаемое слово:
+----+-----+
|word|count|
+----+-----+
|word|    5|
+----+-----+



In [20]:
# Найти самое редко встречаемое слово
least_frequent = word_counts_df.orderBy("count", ascending=True).limit(1)
print("\nСамое редко встречаемое слово:")
least_frequent.show()


Самое редко встречаемое слово:
+--------+-----+
|    word|count|
+--------+-----+
|counting|    1|
+--------+-----+



In [21]:
# Найти среднюю встречаемость слов в тексте
avg_frequency = word_counts_df.select(avg("count")).first()[0]
print(f"\nСредняя встречаемость слов в тексте: {avg_frequency:.2f}")


Средняя встречаемость слов в тексте: 1.45


In [22]:
# Найти общее число уникальных слов
unique_words_count = word_counts_df.count()
print(f"\nОбщее число уникальных слов: {unique_words_count}")


Общее число уникальных слов: 29


In [23]:
# Найти общее число слов в тексте
total_words = words_df.count()
print(f"\nОбщее число слов в тексте: {total_words}")


Общее число слов в тексте: 42


In [24]:
# Закрытие SparkSession
spark.stop()